# 01 — Análise Exploratória de Dados

**Dimensão 3 da rúbrica — 20 pontos.**

Toda visualização precisa de um parágrafo de interpretação logo abaixo.
Gráfico sem leitura perde metade dos pontos do critério.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import numpy as np

# Semente fixa: use em TODO ponto com aleatoriedade (split, modelos, CV).
RANDOM_STATE = 42

APPLICATION = Path("..") / "data" / "raw" / "application_record.csv"          # PREENCHER: nome do arquivo
CREDIT = Path("..") / "data" / "raw" / "credit_record.csv"   
PROCESSED = Path("..") / "data" / "processed"
TARGET = "target"                                            # PREENCHER: variável alvo

pd.set_option("display.max_columns", None)

In [ ]:
application_raw = pd.read_csv(APPLICATION)
credit = pd.read_csv(CREDIT)

## 1. Visão geral

_Dimensões, tipos e primeira impressão do dataset._

#### Histórico de Crédito

In [ ]:
print("Formato do dataset de crédito (qtde de linhas e colunas):", credit.shape)
credit.info()

A coluna "STATUS", que traz o histórico de inadimplência mensal por cliente, é do tipo string. Para facilitar a criação da variável Target, pode ser necessário convertê-la.

In [ ]:
print("\n* Verificação de dados nulos:\n",credit.isnull().sum())
print("\n* Verificação de linhas duplicadas:\n", credit.duplicated().sum())
print("\n* Quantidade de IDs (clientes) únicos:\n", credit['ID'].nunique())

A base de histórico de créditos não contém dados nulos ou linhas duplicadas. 
Vemos também que há apenas 45985 IDs únicos. Como a base tem 1048575 de registros, é possível dizer que, em média, cada cliente conta com um histórico de 23 meses. 

In [ ]:
print ("\nEstatísticas do dataset de crédito:\n")
credit.describe().round(2).T

As estatísticas descritivas trazem informações interessantes:
- o registro de histórico de crédito mais longo é de 60 meses
- o registro de histórico de crédito mais curto é de 1 mês (no caso, o mês atual)
- a média de registro de histórico de crédito é de 19 meses

In [ ]:
sns.countplot(x='STATUS', data=credit)
plt.title('Distribuição dos Status de Crédito')
plt.xlabel('Status de Crédito')
plt.ylabel('Quantidade')
plt.show()

print("\n* Distribuição dos Status de Crédito:\n")
credit['STATUS'].value_counts(normalize=True).round(4)* 100


Observando o dataset 'credit', agrupado por Status de crédito, nota-se que a maior parte dos registros (~62%) aponta para bom pagador (X: sem empréstimos e C: empréstimo pago no mês em análise). Cabe destacar que pode haver clientes repetidos na contagem, pois 1 cliente pode ter longo histórico de crédito.

In [ ]:
#Criação das variáveis TARGET no dataset de crédito, para cada nível de atraso
credit['TARGET0'] = credit['STATUS'].apply(lambda x: 1 if x in ['0','1', '2', '3', '4', '5'] else 0)
credit['TARGET1'] = credit['STATUS'].apply(lambda x: 1 if x in ['1', '2', '3', '4', '5'] else 0)
credit['TARGET2'] = credit['STATUS'].apply(lambda x: 1 if x in ['2', '3', '4', '5'] else 0)
credit['TARGET3'] = credit['STATUS'].apply(lambda x: 1 if x in ['3', '4', '5'] else 0)
credit['TARGET4'] = credit['STATUS'].apply(lambda x: 1 if x in ['4', '5'] else 0)
credit['TARGET5'] = credit['STATUS'].apply(lambda x: 1 if x in ['5'] else 0)

target0 = credit.groupby('ID')['TARGET0'].max().reset_index()
target1 = credit.groupby('ID')['TARGET1'].max().reset_index()
target2 = credit.groupby('ID')['TARGET2'].max().reset_index()
target3 = credit.groupby('ID')['TARGET3'].max().reset_index()
target4 = credit.groupby('ID')['TARGET4'].max().reset_index()
target5 = credit.groupby('ID')['TARGET5'].max().reset_index()

pct_targets = pd.DataFrame({
    "Atraso >= 1 dia": target0['TARGET0'].value_counts(normalize=True).round(4) * 100,
    "Atraso >= 30 dias": target1['TARGET1'].value_counts(normalize=True).round(4) * 100,
    "Atraso >= 60 dias": target2['TARGET2'].value_counts(normalize=True).round(4) * 100,
    "Atraso >= 90 dias": target3['TARGET3'].value_counts(normalize=True).round(4) * 100,
    "Atraso >= 120 dias": target4['TARGET4'].value_counts(normalize=True).round(4) * 100,  
    "Atraso >= 150 dias": target5['TARGET5'].value_counts(normalize=True).round(4) * 100  
}).rename(index={0: "bom pagador (0)", 1: "mau pagador (1)"})

print("\n* Quantidade de IDs (clientes) únicos:\n", credit['ID'].nunique())

print("\n* Distribuição dos dados (em %) de acordo com o nível de atraso:")
display(pct_targets)

Aqui, é trazida a análise diversos níveis para a classificação entre bons e maus pagadores, desde o mais rígido (atraso a partir de 1 dia) até o mais flexível (atraso maior ou igual a 150 dias), considerando o pior status de um cliente ao longo de seus registros de crédito. 

Nesse contexto, é possível observar que para atrasos maiores ou iguais a 30 dias, eleva-se a proporção de clientes indicados como bons pagadores. Nesse sentido, quanto mais flexível o nível de atraso, maior a chance de o modelo de aprendizado de máquina indicar como bom pagador um cliente alta chance de inadimplência. 

Por outro lado, atrasos entre 1 e 29 dias não necessariamente representam um risco iminente de inadimplência e quando adotado um critério mais rígido, o modelo de aprendizado de máquina pode indicar como mau pagador um cliente com baixas chances de inadimplência. 

Dessa forma, para o trabalho de aprendizado de máquina a ser desenvolvido neste projeto, será considerado como risco de aumento de inadimplência clientes com atrasos a partir de 30 dias (categoria 'TARGET1' no dataset 'credit'):
- Mau pagador (TARGET1 = 1): Clientes com algum registro de atraso de 30 dias ou mais (STATUS = 1, 2, 3, 4, 5) em algum mês do histórico de crédito
- Bom pagador (TARGET1 = 0): Clientes sem registro de atraso de 30 dias ou mais (STATUS = 0, C, X) em todos os meses do histórico de crédito

Olhando para a realidade do sistema financeiro atualmente, no geral se considera como crédito inadimplente atrasos > 90 dias. No contexto desta análise, foi considerado um critério mais rígido para maus pagadores, considerando qualquer atraso maior ou igual 30 dias - isso porque  que cartão faz parte de uma linha de crédito sem garantia e com juros elevados. No contexto brasileiro atual, de alto endividamento das famílias e índices de inadimplência em trajetória de crescimento, é importante ser mais conservador na análise, visando evitar elevação das provisões para perda esperada de crédito.

Verifica-se também que o dataframe Target está desbalanceado, sendo necessário refletir sobre técnicas de reamostragem ou ajuste de pesos antes de realizar a modelagem.

#### Cadastro do cliente (application)

In [ ]:
print("Formato do dataset de cadastro (qtde de linhas e colunas):", application_raw.shape)
application_raw.info()

Alguns pontos de atenção nesta primeira visualização:
- Com base no histórico de crédito analisado anteriormente, vimos que há registro para 45985 clientes. No dataset de cadastro, há 438557 clientes. Isso dá indícios de que não há registro de crédito para todos os clientes do cadastro.
- É possível perceber que há dados nulos na coluna "OCCUPATION_TYPE", visto que a quantidade de registros é menor que o total de clientes.
- Há variáveis do tipo string. Neste caso, faz sentido utilizar técnicas de pré-processamento para convertê-las em variáveis numéricas para as análises posteriores.

In [ ]:
print("\n* Verificação de dados nulos:\n", application_raw.isnull().sum())

print("\n* Verificação de linhas duplicadas:\n", application_raw.duplicated().sum())

print("\n* Quantidade de IDs (clientes) únicos:\n", application_raw['ID'].nunique())

print("\n* Quantidade de IDs duplicados:\n", application_raw["ID"].duplicated().sum())


Pontos de atenção:
- Confirmamos que há dados nulos na coluna "OCCUPATION_TYPE". Uma solução possível é considerar o dado da coluna NAME_INCOME_TYPE. Outra alternativa seria marcar como "Desconhecido" os dados faltantes. 
- Verificou-se que a quantidade de IDs únicos é menor que a quantidade de clientes totais do dataset, tendo 47 registros duplicados. Isso indica mau preenchimento do registro, sendo necessário pensar em estratégia sobre como realizar o tratamento. 

In [ ]:
print ("\nEstatísticas do dataset de cadastro:\n")
application_raw.describe().round(2).T

Analisando as estatísticas, são observados alguns pontos de atenção: 
- DAYS_BIRTH: converter para anos, para facilitar a anáise.
- DAYS_EMPLOYED: o valor máximo é de 365243. Pelo dicionário de dados, todos os valores positivos indicam que o cliente está desempregado. Nesse caso, cabe realizar um tratamento dos dados - talvez, indicar 0 para todos os valores positivos. Também vale converter para anos, para facilitar a análise


In [ ]:
application_raw['DAYS_EMPLOYED'].value_counts().sort_index(ascending=False).head()

Aqui, verificamos que a maior parte dos clientes constam como desempregados (DAYS_EMPLOYED > 0)

In [ ]:
application_raw[application_raw['OCCUPATION_TYPE'].isnull()]['NAME_INCOME_TYPE'].value_counts()

Aqui, verificamos como estão distribuídos os grupos por tipo de renda. Na susbstituição de dados nulos da coluna "OCCUPATION_TYPE", será utilizado esse dado como substituto, pois é um indicativo de melhor descrição de ocupação do que "Desconhecido".

In [ ]:
application_raw['NAME_EDUCATION_TYPE'].value_counts()

Aqui, vemos a distribuição dos dados por nível educacional. Para essa variável entrar na matriz de correlação, será necessário convertê-la em valor numérico.

In [ ]:
application_raw['NAME_HOUSING_TYPE'].value_counts()

In [ ]:
#Cruzamento da base de cadastro com a base de target
application_record = application_raw.drop_duplicates(subset="ID", keep="first")
application_merge = pd.merge(application_record, target1, on='ID', how='inner')
application_merge = application_merge.rename(columns={"TARGET1": "TARGET"})

print("\n* Quantidade de clientes únicos com cadastro ANTES do cruzamento:\n", application_raw['ID'].nunique())

print("\n* Quantidade clientes únicos com histórico de crédito:\n", credit['ID'].nunique())

print("\n* Quantidade de clientes únicos com cadastro APÓS o cruzamento:\n", application_merge['ID'].nunique())

print("\n* Clientes descartados da análise, com registro de crédito mas sem cadastro:\n", credit['ID'].nunique() - application_merge['ID'].nunique())

print("\n* Clientes descartados da análise, sem registro de crédito e com cadastro:\n", application_raw['ID'].nunique() - application_merge['ID'].nunique())

print("\n* Distribuição do Target APÓS o cruzamento:\n",application_merge['TARGET'].value_counts(normalize=True).round(4)* 100)
application_merge.head()


Após a definição do target e o cruzamento com a base de cadastro, observa-se que do universo de 438510 com cadastro, apenas 36457 tinham registro de crédito (~8%).

In [32]:
#Ajustando as variáveis DAYS_BIRTH e DAYS_EMPLOYED para anos, criando novas variáveis YEARS_BIRTH e YEARS_EMPLOYED
application_merge['YEARS_BIRTH'] = abs(application_merge['DAYS_BIRTH']) / 365

application_merge['YEARS_EMPLOYED'] = np.where(application_merge['DAYS_EMPLOYED'] < 0,abs(application_merge['DAYS_EMPLOYED']) / 365, 0)

#Adicionando a variável "FLAG_UNEMPLOYED", que indica se o cliente está desempregado (1) ou não (0), com base na variável DAYS_EMPLOYED.
application_merge['FLAG_UNEMPLOYED'] = np.where(application_merge['DAYS_EMPLOYED'] < 0, 0, 1)

#Ajustando a variável OCCUPATION_TYPE, que possui valores nulos, para o valor da variável NAME_INCOME_TYPE, quando OCCUPATION_TYPE for nulo. Caso contrário, mantém o valor original de OCCUPATION_TYPE. Criando uma nova variável chamada OCCUPATION_TYPE_ADJUSTED.
application_merge['OCCUPATION_TYPE_ADJUSTED'] = np.where(application_merge['OCCUPATION_TYPE'].isnull(),application_merge['NAME_INCOME_TYPE'], application_merge['OCCUPATION_TYPE'])

#Ajustandando as variáveis binárias de str para int
application_merge['CODE_GENDER_DUMMY'] = np.where(application_merge['CODE_GENDER'] == 'F', -1, 1)
application_merge['FLAG_OWN_CAR_DUMMY'] = np.where(application_merge['FLAG_OWN_CAR'] == 'Y', 1, 0)
application_merge['FLAG_OWN_REALTY_DUMMY'] = np.where(application_merge['FLAG_OWN_REALTY'] == 'Y', 1, 0)

#Ajustando a variável NAME_EDUCATION_TYPE para valores numéricos
application_merge['NAME_EDUCATION_TYPE_NUM'] = application_merge['NAME_EDUCATION_TYPE'].map(
{   'Lower secondary': 1,
    'Secondary / secondary special': 2,
    'Incomplete higher': 3,
    'Higher education': 4,
    'Academic degree': 5
}
)



In [40]:
application_merge['OCCUPATION_TYPE_ADJUSTED'].value_counts(normalize=True).round(4) * 100

OCCUPATION_TYPE_ADJUSTED
Laborers                 17.04
Pensioner                16.84
Core staff                9.85
Sales staff               9.56
Working                   8.77
Managers                  8.26
Drivers                   5.86
Commercial associate      3.94
High skill tech staff     3.79
Accountants               3.40
Medicine staff            3.31
Cooking staff             1.80
Security staff            1.62
Cleaning staff            1.51
State servant             1.50
Private service staff     0.94
Low-skill Laborers        0.48
Waiters/barmen staff      0.48
Secretaries               0.41
HR staff                  0.23
Realty agents             0.22
IT staff                  0.16
Student                   0.00
Name: proportion, dtype: float64

Olhando a distribuição das profissões, observa-se que as 8 primeiras concentram 80% dos clientes. Dessa forma, vamos fazer ajustes nessas coluna e aplicar a técnica de one hot encoding, transformando as variáveis categócias em valores numéricos


In [41]:
#Agrupando profissisões para aplicação da técnica de one-hot encoding.
professions = [
    "Laborers",
    "Pensioner",
    "Core staff",
    "Sales staff",
    "Working",
    "Managers",
    "Drivers",
    "Commercial associate"
]

application_merge['OCCUPATION_TYPE_GROUPED'] = np.where(
    application_merge['OCCUPATION_TYPE_ADJUSTED'].isin(professions),
    application_merge['OCCUPATION_TYPE_ADJUSTED'],
    "Others"
)



In [ ]:
application_merge.head(1).T

## 2. Distribuição das variáveis

Neste passo, será considerado o dataset application_merge, que já contém a variável target para os 36457 clientes. 

#### Histograma & boxplot das variáveis numéricas do dataset de cadastro


In [ ]:
#Renda anual
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(application_merge['AMT_INCOME_TOTAL'], bins=10, edgecolor='black')
ax[0].set_ylabel('Frequência')
ax[0].set_xlabel('Renda anual')
ax[0].set_title('Histograma da variável renda anual (AMT_INCOME_TOTAL)')

ax[1].boxplot(application_merge['AMT_INCOME_TOTAL'])
ax[1].set_ylabel('Renda anual')
ax[1].set_title('Boxplot da variável renda anual (AMT_INCOME_TOTAL)')

plt.tight_layout()
plt.show()

print("Maior frequência:", application_merge['AMT_INCOME_TOTAL'].value_counts().head(1))
print("Maior Valor:",application_merge['AMT_INCOME_TOTAL'].max())

Quanto à renda anual, o histograma indica uma curva assimétrica à direita, com a maior quantidade concentrada em $135000. No boxplot, são observados também outliers - destaque para o maior valor de renda anual, de $1575000.



In [ ]:
#Idade
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(application_merge['YEARS_BIRTH'], bins=10, edgecolor='black')
ax[0].set_ylabel('Frequência')
ax[0].set_xlabel('Idade')
ax[0].set_title('Histograma da variável idade (YEARS_BIRTH)')

ax[1].boxplot(application_merge['YEARS_BIRTH'])
ax[1].set_ylabel('Idade')
ax[1].set_title('Boxplot da variável idade (YEARS_BIRTH)')

plt.tight_layout()
plt.show()

print("Maior frequência:", application_merge['YEARS_BIRTH'].value_counts().head(3))
print("Maior Valor:",application_merge['YEARS_BIRTH'].max())

Quanto à idade, o histograma aparenta uma distribuição normal, com concentração maior entre 34 e 46 anos. No boxplot, não observamos outliers.

In [ ]:
#Tempo de serviço
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(application_merge['YEARS_EMPLOYED'], bins=np.arange(0, application_merge['YEARS_EMPLOYED'].max() + 1), edgecolor='black')
ax[0].set_ylabel('Frequência')
ax[0].set_xlabel('Tempo de serviço (anos)')
ax[0].set_title('Histograma da variável tempo de serviço (YEARS_EMPLOYED)')

ax[1].boxplot(application_merge['YEARS_EMPLOYED'])
ax[1].set_ylabel('Tempo de serviço (anos)')
ax[1].set_title('Boxplot da variável tempo de serviço (YEARS_EMPLOYED)')

plt.tight_layout()
plt.show()

print("Maior frequência:", application_merge['YEARS_EMPLOYED'].value_counts().head(1))
print("Maior Valor:",application_merge['YEARS_EMPLOYED'].max())

Em relação ao tempo de serviço, pelo histograma nota-se uma curva assimétrica à direita, com a maior parte dos clientes como desempregado (YEARS_EMPLOYED = 0), que pode indicar risco à inadimplência.
No boxplot, são observados outliers também, sendo 43 anos o maior tempo de serviço.

In [ ]:
#Número de filhos
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(application_merge['CNT_CHILDREN'], bins=np.arange(0, application_merge['CNT_CHILDREN'].max() + 1), edgecolor='black')
ax[0].set_ylabel('Frequência')
ax[0].set_xlabel('Número de filhos')
ax[0].set_title('Histograma da variável número de filhos (CNT_CHILDREN)')

ax[1].boxplot(application_merge['CNT_CHILDREN'])
ax[1].set_ylabel('Número de filhos')
ax[1].set_title('Boxplot da variável número de filhos (CNT_CHILDREN)')

plt.tight_layout()
plt.show()

print("Maior frequência:", application_merge['CNT_CHILDREN'].value_counts().head(1))
print("Maior Valor:",application_merge['CNT_CHILDREN'].max())

Sobre o número de filhos, o histograma mostra uma curva assimétrica à direita, com a maioria dos clientes sem filhos. No boxplot, observamos alguns outliers, sendo o maior 19 filhos - isso pode ser indício de um registro mal feito. 

In [ ]:
#Número de membros da família
fig, ax = plt.subplots(1, 2, figsize=(14, 5))

ax[0].hist(application_merge['CNT_FAM_MEMBERS'], bins=np.arange(0, application_merge['CNT_FAM_MEMBERS'].max() + 1), edgecolor='black')
ax[0].set_ylabel('Frequência')
ax[0].set_xlabel('Número de membros da família')
ax[0].set_title('Histograma da variável número de membros da família (CNT_FAM_MEMBERS)')

ax[1].boxplot(application_merge['CNT_FAM_MEMBERS'])
ax[1].set_ylabel('Número de membros da família')
ax[1].set_title('Boxplot da variável número de membros da família (CNT_FAM_MEMBERS)')

plt.tight_layout()
plt.show()

print("Maior frequência:", application_merge['CNT_FAM_MEMBERS'].value_counts().head(1))
print("Maior Valor:",application_merge['CNT_FAM_MEMBERS'].max())

Sobre o tamanho da família, observa-se que é uma curva assimétrica à direita, com a maior parte das famílias compostas por 2 membros. O boxplot aponta outliers também, sendo a família mais numerosa com 20 membros. 

#### Gráficos das variáveis categóricas do dataset de cadastro

In [ ]:
fig, axes = plt.subplots(4, 1, figsize=(12,25))

#axes = ax.flatten()

sns.countplot(x='CODE_GENDER', data=application_merge, ax=axes[0], order=application_merge['CODE_GENDER'].value_counts().index)
axes[0].set_ylabel('Frequência')
axes[0].set_xlabel('Gênero')
axes[0].set_title('Distribuição de clientes por gênero (CODE_GENDER)')

sns.countplot(x='NAME_EDUCATION_TYPE', data=application_merge, ax=axes[1], order=application_merge['NAME_EDUCATION_TYPE'].value_counts().index)
axes[1].set_ylabel('Frequência')
axes[1].set_xlabel('Nível educacional')
axes[1].set_title('Distribuição de clientes por nível educacional (NAME_EDUCATION_TYPE)')
axes[1].tick_params(axis='x', rotation=15)

sns.countplot(x='NAME_FAMILY_STATUS', data=application_merge, ax=axes[2], order=application_merge['NAME_FAMILY_STATUS'].value_counts().index)
axes[2].set_ylabel('Frequência')
axes[2].set_xlabel('Estado civil')
axes[2].set_title('Distribuição de clientes por estado civil (NAME_FAMILY_STATUS)')
axes[2].tick_params(axis='x', rotation=15)


sns.countplot(x='OCCUPATION_TYPE_ADJUSTED', data=application_merge, ax=axes[3], order=application_merge['OCCUPATION_TYPE_ADJUSTED'].value_counts().index)
axes[3].set_ylabel('Frequência')
axes[3].set_xlabel('Tipo de ocupação')
axes[3].set_title('Distribuição de clientes por tipo de ocupação (OCCUPATION_TYPE_ADJUSTED)')
axes[3].tick_params(axis='x', rotation=60)

plt.tight_layout()
plt.show()

print(application_merge['CODE_GENDER'].value_counts(normalize=True).round(4) * 100, "\n")
print(application_merge['NAME_EDUCATION_TYPE'].value_counts(normalize=True).round(4) * 100, "\n")
print(application_merge['NAME_FAMILY_STATUS'].value_counts(normalize=True).round(4) * 100, "\n")
print(application_merge['OCCUPATION_TYPE_ADJUSTED'].value_counts(normalize=True).round(4) * 100, "\n")

Em relação ao gênero, 67% dos cllientes é do sexo feminino. 
Quanto ao nível educacional, 68% possui Ensino Médio (Secondary/Secondary special).
Sobre o estado civil, observa-se que 69% dos clientes da base é composta de pessoas casadas - o que se relaciona com o tamanho das famílias visto no histograma, composta por 2 pessoas. 
Quanto ao tipo de ocupação, observa-se maior concentração de clientes como Trabalhadores (Laborers) e Pensionistas/Aposentados (Pensioners). 


#### Análise das features com Target

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(10, 18))

sns.boxplot(x='TARGET', y='AMT_INCOME_TOTAL', data=application_merge, palette = 'hls', ax=axes[0])
axes[0].set_title('Renda anual (AMT_INCOME_TOTAL) x Target')
axes[0].set_xlabel('Target (0 = Bom pagador, 1 = Mau pagador)')
axes[0].set_ylabel('Renda Anual')

sns.boxplot(x='TARGET', y='YEARS_BIRTH', data=application_merge, palette = 'hls', ax=axes[1])
axes[1].set_title('Idade (YEARS_BIRTH) x Target')
axes[1].set_xlabel('Target (0 = Bom pagador, 1 = Mau pagador)')
axes[1].set_ylabel('Idade')

sns.boxplot(x='TARGET', y='YEARS_EMPLOYED', data=application_merge, palette = 'hls', ax=axes[2])
axes[2].set_title('Tempo de Emprego (YEARS_EMPLOYED) x Target')
axes[2].set_xlabel('Target (0 = Bom pagador, 1 = Mau pagador)')
axes[2].set_ylabel('Tempo de Emprego')

plt.tight_layout()
plt.show()

Sobre a renda anual do cliente, observa-se que a distribuição entre as classes é similar. Isso pode trazer um indício de que a renda, isoladamente, não é um preditor adequado para classificar o cliente como bom ou mau pagador. 
Em relação à idade, observa-se para maus pagadores, a mediana é um pouco menor quando comparada aos bons pagadores. Nesse contexto, é possível inferir que maus pagadores tendem a ser mais jovens em relação a bons pagadores. 
Quanto ao tempo de serviço, também se observa uma distribuição similar entre as cllasses.

#### Análise de variáveis categóricas com Target

In [ ]:
pd.crosstab(application_merge['CODE_GENDER'], application_merge['TARGET'], normalize='index') * 100

Em relação ao gênero, observa-se que no gênero masculino, a proporção de maus pagadores é maior do que no gênero feminino.

In [ ]:
pd.crosstab(application_merge['NAME_EDUCATION_TYPE'], application_merge['TARGET'], normalize='index') * 100

Em relação ao nível educional, observa-se que dentre o clientes com pós-graduação (Academic degree) observa-se uma proporção maior de maus pagadores (~22%)

In [ ]:
pd.crosstab(application_merge['NAME_FAMILY_STATUS'], application_merge['TARGET'], normalize='index') * 100

Pelo estado civil, observa-se que o grupo de solteiros/não casados conta com a maior proporção de maus pagadores (~13%)

In [ ]:
pd.crosstab(application_merge['OCCUPATION_TYPE_ADJUSTED'], application_merge['TARGET'], normalize='index') * 100

In [ ]:
application_merge['OCCUPATION_TYPE_ADJUSTED'].value_counts().sort_values(ascending=True).head(10)

Em relação à ocupação, o grupo de estudantes tem uma proporção de 100% de mau pagador. No entanto, cabe ressaltar que temos apenas 1 cliente como estudante no dataset application_merge, uma amostra pequena diante dos mais de 36 mil clientes. 

## 3. Correlações

**Interpretação:** _comente cada correlação relevante, uma a uma._

In [ ]:
# heatmap de correlação

In [ ]:
application_merge.info()

In [ ]:
cols = [
    "TARGET",
    "CODE_GENDER_DUMMY",
    "YEARS_BIRTH",
    "YEARS_EMPLOYED",
    "NAME_EDUCATION_TYPE_NUM",
    "CNT_CHILDREN",
    "CNT_FAM_MEMBERS",
    "AMT_INCOME_TOTAL",
    "FLAG_OWN_CAR_DUMMY",
    "FLAG_OWN_REALTY_DUMMY",
    "FLAG_WORK_PHONE",
    "FLAG_PHONE",
    "FLAG_EMAIL",
    "FLAG_UNEMPLOYED",
]

plt.figure(figsize=(14, 12))
sns.heatmap(
    application_merge[cols].corr(),
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    square=True,
    annot_kws={"size": 10},
    cbar_kws={"shrink": .8}
)

plt.tight_layout()
plt.show()

O mapa de calor traz aspectos interessantes:
- forte correlação positiva entre quantidade de filhos (CNT_CHILDREN) e quantidade de membros da família (CNT_FAM_MEMBERS). Para mitigar vieses na modelagem, cabe excluir uma dessas variáveis da análise.
- também se observa correlação negativa entre número de filhos (CNT_CHILDREN) e idade do cliente (YEARS_BIRTH). Pode ser uma solução desconsiderar a quantidade de filjps do modelo.

## 4. Outliers

**Interpretação:** _remover, winsorizar ou manter? Justifique._

In [ ]:
# detecção de outliers
application_merge.describe().round(2).T

## 5. Balanceamento de classes

**Interpretação:** _qual a proporção entre as classes e o que isso implica na escolha das métricas?_

In [ ]:
print("\n* Distribuição do Target ANTES do cruzamento de bases:\n",target1['TARGET1'].value_counts(normalize=True).round(4) * 100)


print("\n* Distribuição do Target APÓS o cruzamento (dataset application_merge):\n",application_merge['TARGET'].value_counts(normalize=True).round(4)* 100)

Analisando a variável target, observa-se um desbalanceamento das classes, com ~12% dos clientes considerados mau pagadores e ~88% como bom pagadores.
Dado esse desbalanceamento, a acurácia não será a melhor métrica para avaliação dos modelos de aprendizado de máquina. Na avaliação, será necessário olhar também para Recall, F1 Score e Matriz de Confusão. 
Também será necessário pensar em técnicas para contornar esse desbalanceamento das classes.

## 6. Síntese da EDA

_Amarre os achados em uma narrativa única — é ela que vira o storytelling da apresentação._